# Carnot Heat Pump and Refrigeration

**Learning outcome:** Apply carnot heat pump and refrigeration through the public `PinchProblem` or `PinchWorkspace` workflow.

**Level:** Intermediate  
**Execution profile:** `slow-hpr`  
**Expected runtime:** 2 to 10 minutes  
**Optional extras:** hpr

The lifecycle is explicit: prepare the study, run the named method, then inspect cached results. Observation cells do not launch analysis.

## Study question and data

**Study question:** Where can idealized heat pumping or refrigeration reduce utility demand before detailed equipment selection?

The sample data is packaged with OpenPinch, so the notebook runs without path setup. Read the named inputs and assumptions before substituting plant data.

## Step 1: Screen a process heat pump

Run this cell once, then inspect its named outputs. Arguments on the method call apply to this analysis; stored configuration is only the fallback when an argument is omitted.

In [ ]:
from OpenPinch import PinchProblem

problem = PinchProblem("heat_pump_targeting.json", project_name="Heat Pump Study")
# Ratios to electricity price affect Carnot screening; utility stream prices
# affect the subsequent utility allocation. Keep global defaults unchanged.
economics = {"COSTING_HPR_PRICE_RATIO_HEAT_TO_ELE": 1.0,
             "COSTING_HPR_PRICE_RATIO_COLD_TO_ELE": 0.1}
heat_pump = problem.target.carnot_heat_pump(
    is_utility_heat_pump=False,
    is_cascade_cycle=True,
    load_fraction=0.25,
    condensers=1,
    evaporators=1,
    maximum_restarts=1,
    options=economics,
)
hpr_summary = problem.summary_frame()
load_hp_plot = problem.plot.net_load_profiles_with_heat_pump(target=heat_pump)
gcc_hp_plot = problem.plot.grand_composite_curve_with_heat_pump(target=heat_pump)
print(heat_pump.hpr_load.model_dump())


## Step 2: Screen refrigeration and inspect curves

Run this cell once, then inspect its named outputs. Arguments on the method call apply to this analysis; stored configuration is only the fallback when an argument is omitted.

In [ ]:
# Compare modes with the same process basis and stage counts.
refrigeration = problem.target.carnot_refrigeration(
    is_utility_refrigeration=False,
    is_cascade_cycle=True,
    load_fraction=0.25,
    condensers=1,
    evaporators=1,
    maximum_restarts=1,
    options=economics,
)
refrigeration_summary = problem.summary_frame()
load_rfgn_plot = problem.plot.net_load_profiles_with_refrigeration(target=refrigeration)
gcc_rfgn_plot = problem.plot.grand_composite_curve_with_refrigeration(target=refrigeration)
print(refrigeration.hpr_load.model_dump())


## Step 3: Allocate and place residual utilities

Run this cell once, then inspect its named outputs. Arguments on the method call apply to this analysis; stored configuration is only the fallback when an argument is omitted.

In [ ]:
# Allocate the existing utilities on the fixed heat-pump residual.
utilities = problem.target.all_heat_integration(base_target=heat_pump)
optimized = problem.target.utility_placement(
    base_target=heat_pump,
    isothermal=2,
    options={"iteration_limit": 20, "evaluation_limit": 200, "seed": 20260715},
)
placement_summary = optimized.summary_frame()
optimized_gcc = optimized.plot.grand_composite_curve()
# Explicit case derivation is also available and does not solve.
residual = problem.residual_utility(base_target=heat_pump)
# Transfer utilities to a new ORIGINAL-process study; this does not install HPR.
new_problem = problem.with_utilities_from(optimized)
new_results = new_problem.target.all_heat_integration()
assert optimized.to_problem_json()["residual_basis"] == residual.to_problem_json()["residual_basis"]


## Review the result

Inspect separate target-specific plots and both duty summaries. Residual utility placement is sequential: HPR duty and temperatures remain fixed. It does not jointly optimize the heat pump and utilities. Changing utility prices later does not resize HPR.

In [ ]:
from IPython.display import display

display(hpr_summary)
display(refrigeration_summary)
display(load_hp_plot)
display(gcc_hp_plot)
display(load_rfgn_plot)
display(gcc_rfgn_plot)
display(utilities)
display(placement_summary)

## Interpret the result

load_fraction selects a fraction of available net heating (heat pump) or net cooling (refrigeration), in [0, 1]. It is not compressor part-load. Heat pumps may use less than the selected ceiling when economics favour utilities. Refrigeration targets the selected cooling service. Inspect available, selected and achieved duty separately from ambient exchange. The explicit positive cold/electricity price ratio is 0.1 here; the global default 1.0 can favour more cooling displacement. Utility stream prices do not set this screening ratio. For the same candidate leaving 100 kW of cooling, the cooling-cost contribution falls from 100 to 10 kW electricity-equivalent as this ratio falls from 1.0 to 0.1. This changes cost without adding a heat-pump feasibility penalty.

## Adapt this template

Vary load fraction and utility placement deliberately, then carry promising duties into a simulated-cycle study.

Keep the workflow explicit: prepare input, call one named engineering method, inspect cached results, then export.